# Digital Twins – Week 2: Household Energy Consumption
## System: Electricity consumption of household appliances in a residential building.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('energydata_complete.csv')
df.head()

In [ ]:
print('Number of rows:', len(df))
print('Columns:', list(df.columns))
print(df[['date', 'Appliances', 'T1', 'RH_1', 'T_out', 'RH_out']].head())

# 4. Clean the data step by step

Remove missing values and zero values from appliance consumption

In [ ]:
df = df[['date', 'Appliances', 'T1', 'RH_1', 'T_out', 'RH_out']].copy()
df['date'] = pd.to_datetime(df['date'])
#Remove missing values
df = df.dropna()
#Remove zero appliance consumptions
df = df[df['Appliances'] > 0]
print('Remaining rows:', len(df))

## 4.a Filter unrealistic humidity values

Should be between 0% and 100%. Using a range of 5%-95% as a rule

In [ ]:
df = df[(df['RH_1'] >= 5) & (df['RH_1'] <= 95)]
df = df[(df['RH_out'] >= 5) & (df['RH_out'] <= 95)]
print('Remaining rows:', len(df))

## 4.b Filter unrealistic temperature valuse
* For indoors: 10-30 °C
* Outdoors: -20-45 °C

In [ ]:
df = df[(df['T1'] >= 10) & (df['T1'] <= 30)]
df = df[(df['T_out'] >= -20) & (df['T_out'] <= 45)]
print('Remaining rows:', len(df))

## Selecting a date


In [ ]:
df = df.sort_values('date')
day = df[df['date'].dt.date == df['date'].dt.date.iloc[42]].copy()
print('Selected day:', day['date'].iloc[0].date())
print('Measurements:', len(day))

# 5. Define our simulation function

**Question** : How would appliance energy consumption change if devices were more efficient or if high consumption were limited


In [ ]:
def simulate_consumption(values, saving_percent=0, max_energy=None):
    results = []
    for consumption in values:
        new_value = consumption * (1 - saving_percent / 100)
        if max_energy is not None and new_value > max_energy:
            new_value = max_energy
        results.append(new_value)
    return results

# 6. Baseline and scenarios
**1. Baseline**: no changes.

**2. Efficient appliances**: 15% lower consumption.

**3. Peak limit**: maximum 150 Wh per 10 minutes.

**4. Combined scenario**: efficient appliances + peak limit.

In [ ]:
baseline = simulate_consumption(day['Appliances'])
efficient = simulate_consumption(day['Appliances'], saving_percent=15)
peak_limit = simulate_consumption(day['Appliances'], max_energy=150)
combined = simulate_consumption(day['Appliances'], saving_percent=15, max_energy=150)

# 7. Charts - baseline

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(day['date'], baseline, label='Measured consumption')
plt.title('Baseline – appliance energy consumption')
plt.xlabel('Time')
plt.ylabel('Energy per 10 minutes (Wh)')
plt.legend()
plt.grid()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 8. Chart - baseline vs first scenario

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(day['date'], baseline, label='Baseline')
plt.plot(day['date'], efficient, label='15% energy saving')
plt.title('Baseline vs. efficient appliances')
plt.xlabel('Time')
plt.ylabel('Energy per 10 minutes (Wh)')
plt.legend()
plt.grid()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 9. Chart - baseline vs peak max

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(day['date'], baseline, label='Baseline')
plt.plot(day['date'], peak_limit, label='Peak limit: 150 Wh')
plt.title('Baseline vs. efficient appliances')
plt.xlabel('Time')
plt.ylabel('Energy per 10 minutes (Wh)')
plt.legend()
plt.grid()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 10. Chart - compare all scenarios

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(day['date'], baseline, label='Baseline')
plt.plot(day['date'], efficient, label='15% energy saving')
plt.plot(day['date'], peak_limit, label='Peak limit: 150 Wh')
plt.plot(day['date'], combined, label='Combined scenario')
plt.title('Comparison of energy consumption scenarios')
plt.xlabel('Time (hours of the day)')
plt.ylabel('Energy per 10 minutes (Wh)')
plt.legend()
plt.grid()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 11. Compare total energy consumption

Appliances values are energy amounts (Wh) for 10-minute intervals, so summing them gives the total energy for the selected day. Divide by 1,000 to convert Wh to kWh.

In [ ]:
print('Baseline:', round(sum(baseline) / 1000, 2), 'kWh')
print('15% energy saving:', round(sum(efficient) / 1000, 2), 'kWh')
print('Peak limit:', round(sum(peak_limit) / 1000, 2), 'kWh')
print('Combined scenario:', round(sum(combined) / 1000, 2), 'kWh')